In [0]:
from pyspark.sql.functions import current_date, lit

df_init = spark.createDataFrame([
    (1, "Alice", 1000),
    (2, "Bob", 2000)
], ["id", "name", "salary"])

df_init = df_init \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

df_init.write.mode("append").saveAsTable("workspace.default.employees_scd2")

In [0]:
df_updates = spark.createDataFrame([
    (1, "Alice", 1500),  # salary changed
    (3, "Charlie", 3000) # new record
], ["id", "name", "salary"])

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date

target = DeltaTable.forName(spark, "workspace.default.employees_scd2")

# Step 1: Expire old records
target.alias("t").merge(
    df_updates.alias("s"),
    "t.id = s.id AND t.is_current = true"
).whenMatchedUpdate(
    condition="t.salary <> s.salary",
    set={
        "end_date": "current_date()",
        "is_current": "false"
    }
).execute()